# Phase 3: Build a Strong Retrieval Pipeline

## Step 10: Metadata, Filtering, and Citations

### Learning

- Metadata design
- Source provenance
- Document-level filtering
- Access control
- Source citations
- Document versions
- Data freshness
- Grounded generation

---

## Key Takeaways

- Retrieval is not just "find similar text" — production systems need to filter by
  **who is asking** and **which document is authoritative**, before anything is sent
  to the model. Access control is a retrieval-time concern, not a prompting concern.
- A citation marker in an answer is not proof the answer is correct. The model can
  invent `[7]` out of thin air, or cite a real source that doesn't actually support
  the sentence next to it. Both need to be checked with code, not assumed.
- "Newest" and "most relevant" are different axes. Hybrid search (Step 7) already
  taught us not to mix incompatible score scales — freshness scoring hits the exact
  same trap, so we normalize before blending, same as Step 7 Section 5.
- Some documents should *not* be freshness-boosted (a 2019 policy that is still the
  current authority). Freshness is a preference, not a blind sort key.
- When two active-looking chunks disagree (a policy changed and both versions got
  retrieved), the right behavior is to surface the conflict — not silently pick one
  or average the numbers together.

---

## To do (mirrors the Roadmap 1:1)

1. Define a metadata schema
2. Add metadata filters to retrieval
3. Enforce user access before generation
4. Add source numbers
5. Require citations in generated answers
6. Render citations in the interface
7. Validate citation references
8. Add source-support checking
9. Handle document versions
10. Add freshness rules
11. Add insufficient-evidence behavior

**A note on scope:** this notebook builds directly on the retrieval mechanics from
Steps 7 (hybrid search / RRF) and 8 (reranking). It reuses the hybrid-search plumbing
from Step 7 (which is fully implemented) but does **not** re-run the Step 8 reranker —
Step 8's reranking cells are still learner TODOs in this repo, and reranking is
orthogonal to the governance layer this notebook adds. Everything below applies
identically whether or not a reranking stage sits between fusion and citation
assignment — just insert it after Section 3 (fused, access-filtered candidates) and
before Section 4 (source numbering).

## 0. Environment Setup

Same connection pattern as every step since Step 4: Elasticsearch for lexical
search, Chroma for vector search, OpenAI for embeddings and generation. This
notebook uses its own index/collection so it doesn't collide with Step 7-9's data.

We also define a fictional multi-document **ByteMage** corpus — continuing the
same company/persona (`John Doe`, `Sarah Ahmed`, `AI Search Platform`) introduced
in `sample.md` back in Step 7/9 — because metadata filtering, access control, and
version conflicts are hard to demonstrate convincingly with a single-document
biography. A real internal knowledge base has many documents, owners,
departments, and versions; this notebook simulates that.

In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

try:
    response = es.info()
    print(response)
except Exception as e:
    print("TYPE:", type(e).__name__)
    print("ERROR:")
    print(e)
    if hasattr(e, "body"):
        print("\nBODY:")
        print(e.body)

import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import config
importlib.reload(config)

{'name': '0e1f447b24d9', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'lHCCr2OyQ0yMLKjGmwdmBg', 'version': {'number': '8.19.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '93788a8c2882eb5b606510680fac214cff1c7a22', 'build_date': '2025-07-23T22:10:18.138212839Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

In [2]:
import hashlib
import re
import time
from datetime import datetime, timezone

import chromadb
from openai import OpenAI
from pydantic import BaseModel, Field

from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma = chromadb.HttpClient(host="localhost", port=8000)

INDEX_NAME = "rag_documents_governed"
COLLECTION_NAME = "bytemage_governed_docs"

NOW = datetime.now(timezone.utc)
print("Notebook run timestamp (UTC):", NOW.isoformat())

Notebook run timestamp (UTC): 2026-08-18T11:26:18.498583+00:00


## 1. Define a Metadata Schema

> Every chunk should include fields such as: `chunk_id`, `document_id`, `title`,
> `source`, `page`, `section`, `version`, `created_at`, `owner`, `access_groups`,
> `content_hash`. Use consistent data types across Chroma, Elasticsearch, and the
> relational database.

We extend the roadmap's example schema with a few fields the later sections need:

| field | type | purpose |
|---|---|---|
| `chunk_id` / `document_id` | string | stable identifiers |
| `title`, `section`, `source` | string | what to show in a citation |
| `department`, `owner` | string | filtering |
| `version`, `status` | string | `status` is `"active"` or `"archived"` (Section 9) |
| `created_at` / `created_at_ts` | ISO string / int | freshness scoring needs a sortable number (Section 10) |
| `access_groups` | list[string] | access control (Section 3) |
| `time_sensitive` | bool | should this doc be freshness-boosted at all? |
| `authoritative` | bool | exempt from freshness penalties (Section 10) |
| `content_hash` | string | detect duplicate/unchanged content, same idea as Step 7 Section 6 |

**Elasticsearch** stores `access_groups` natively as a keyword array — a `terms`
query matches if *any* element overlaps. **Chroma** metadata values must be
scalars (str/int/float/bool), so list-valued fields like `access_groups` get
flattened into one boolean column per group (`access_employees`,
`access_finance`, ...) when we index into Chroma below. Both representations
carry the same information; this is exactly the "use consistent data types
across stores" requirement — consistent *meaning*, store-appropriate
*encoding*.

In [3]:
def content_hash(text):
    """Stable hash of normalized chunk text — same idea as Step 7 Section 6."""
    normalized = " ".join(text.lower().split())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def make_chunk(
    chunk_id,
    document_id,
    title,
    source,
    section,
    department,
    owner,
    version,
    status,
    created_at,
    access_groups,
    text,
    time_sensitive=False,
    authoritative=False,
):
    """Build one chunk in the common metadata schema described above."""
    created_at_ts = int(datetime.fromisoformat(created_at).replace(tzinfo=timezone.utc).timestamp())

    return {
        "chunk_id": chunk_id,
        "document_id": document_id,
        "title": title,
        "source": source,
        "section": section,
        "department": department,
        "owner": owner,
        "version": version,
        "status": status,
        "created_at": created_at,
        "created_at_ts": created_at_ts,
        "access_groups": access_groups,
        "time_sensitive": time_sensitive,
        "authoritative": authoritative,
        "text": text,
        "content_hash": content_hash(text),
    }

In [4]:
# A small fictional ByteMage knowledge base. Two documents intentionally have
# two versions each, to drive Section 9 (versions) and Section 10 (freshness):
#   - "leave-policy"    -> archived v1 vs active v2, CONTRADICTORY sick-day counts
#   - "product-roadmap" -> two active versions, same topic, only recency differs
bytemage_documents = [
    make_chunk(
        chunk_id="leave-policy-v1-001",
        document_id="leave-policy",
        title="ByteMage Leave Policy",
        source="leave-policy-v1.md",
        section="Sick Leave",
        department="HR",
        owner="HR",
        version="2023-02",
        status="archived",
        created_at="2023-02-01",
        access_groups=["employees"],
        text=(
            "ByteMage employees may take up to three sick days per month without "
            "additional approval. Extended sick leave requires manager sign-off."
        ),
    ),
    make_chunk(
        chunk_id="leave-policy-v2-001",
        document_id="leave-policy",
        title="ByteMage Leave Policy",
        source="leave-policy-v2.md",
        section="Sick Leave",
        department="HR",
        owner="HR",
        version="2025-09",
        status="active",
        created_at="2025-09-01",
        access_groups=["employees"],
        text=(
            "ByteMage employees may take up to five sick days per month without "
            "additional approval. Extended sick leave requires notifying HR within "
            "48 hours."
        ),
    ),
    make_chunk(
        chunk_id="compensation-policy-001",
        document_id="compensation-policy",
        title="ByteMage Compensation Policy",
        source="compensation-policy.md",
        section="Salary Bands",
        department="Finance",
        owner="Finance",
        version="2025-04",
        status="active",
        created_at="2025-04-01",
        access_groups=["finance", "leadership"],
        text=(
            "ByteMage salary bands are reviewed every March. Senior Software "
            "Engineers in the AI Platform department fall in Band E5."
        ),
    ),
    make_chunk(
        chunk_id="data-retention-policy-001",
        document_id="data-retention-policy",
        title="ByteMage Data Retention Policy",
        source="data-retention-policy.md",
        section="Customer Data",
        department="Security",
        owner="Security",
        version="2019-05",
        status="active",
        created_at="2019-05-01",
        access_groups=["employees"],
        authoritative=True,
        text=(
            "ByteMage retains customer support data for a minimum of seven years "
            "under regulatory requirement RX-118. This requirement has not changed "
            "since 2019."
        ),
    ),
    make_chunk(
        chunk_id="engineering-handbook-001",
        document_id="engineering-handbook",
        title="ByteMage Engineering Handbook",
        source="engineering-handbook.md",
        section="Code Review Standards",
        department="Engineering",
        owner="Engineering",
        version="2025-07",
        status="active",
        created_at="2025-07-15",
        access_groups=["employees", "engineering"],
        text=(
            "ByteMage pull requests require at least one approving review from a "
            "senior engineer before merging to main. John Doe co-authored this "
            "standard as part of the AI Platform team's review guidelines."
        ),
    ),
    make_chunk(
        chunk_id="product-roadmap-2025-001",
        document_id="product-roadmap",
        title="ByteMage Product Roadmap",
        source="product-roadmap-2025.md",
        section="Q1 Priorities",
        department="Product",
        owner="Product",
        version="2025-01",
        status="active",
        created_at="2025-01-05",
        access_groups=["employees"],
        time_sensitive=True,
        text=(
            "As of Q1 2025, ByteMage's top product priority is migrating internal "
            "services to the new authentication system."
        ),
    ),
    make_chunk(
        chunk_id="product-roadmap-2026-001",
        document_id="product-roadmap",
        title="ByteMage Product Roadmap",
        source="product-roadmap-2026.md",
        section="Q3 Priorities",
        department="Product",
        owner="Product",
        version="2026-07",
        status="active",
        created_at="2026-07-20",
        access_groups=["employees"],
        time_sensitive=True,
        text=(
            "As of Q3 2026, ByteMage's top product priority is launching the AI "
            "Search Platform's new billing dashboard for enterprise customers."
        ),
    ),
    make_chunk(
        chunk_id="onboarding-guide-001",
        document_id="onboarding-guide",
        title="ByteMage Onboarding Guide",
        source="onboarding-guide.md",
        section="First Week",
        department="HR",
        owner="HR",
        version="2024-01",
        status="active",
        created_at="2024-01-10",
        access_groups=["employees"],
        text=(
            "New ByteMage employees complete orientation during their first week, "
            "including IT setup, benefits enrollment, and an introduction to the "
            "AI Search Platform."
        ),
    ),
]

print(f"{'chunk_id':<26} {'doc_id':<20} {'version':<9} {'status':<9} access_groups")
print("-" * 90)
for chunk in bytemage_documents:
    print(
        f"{chunk['chunk_id']:<26} {chunk['document_id']:<20} "
        f"{chunk['version']:<9} {chunk['status']:<9} {chunk['access_groups']}"
    )

chunk_id                   doc_id               version   status    access_groups
------------------------------------------------------------------------------------------
leave-policy-v1-001        leave-policy         2023-02   archived  ['employees']
leave-policy-v2-001        leave-policy         2025-09   active    ['employees']
compensation-policy-001    compensation-policy  2025-04   active    ['finance', 'leadership']
data-retention-policy-001  data-retention-policy 2019-05   active    ['employees']
engineering-handbook-001   engineering-handbook 2025-07   active    ['employees', 'engineering']
product-roadmap-2025-001   product-roadmap      2025-01   active    ['employees']
product-roadmap-2026-001   product-roadmap      2026-07   active    ['employees']
onboarding-guide-001       onboarding-guide     2024-01   active    ['employees']


### Index the corpus into Elasticsearch and Chroma

Same pattern as Steps 4-7, extended with the metadata fields above. `access_groups`
is stored as an ES keyword array; in Chroma it is flattened into one boolean field
per group, as explained in Section 1.

In [5]:
# ---- Elasticsearch (lexical side) ----
from elasticsearch.helpers import bulk

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

mapping = {
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "document_id": {"type": "keyword"},
            "title": {"type": "text"},
            "text": {"type": "text"},
            "source": {"type": "keyword"},
            "section": {"type": "keyword"},
            "department": {"type": "keyword"},
            "owner": {"type": "keyword"},
            "version": {"type": "keyword"},
            "status": {"type": "keyword"},
            "created_at": {"type": "date"},
            "created_at_ts": {"type": "long"},
            "access_groups": {"type": "keyword"},
            "time_sensitive": {"type": "boolean"},
            "authoritative": {"type": "boolean"},
            "content_hash": {"type": "keyword"},
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)

actions = [
    {"_index": INDEX_NAME, "_id": chunk["chunk_id"], "_source": chunk}
    for chunk in bytemage_documents
]
success, failed = bulk(es, actions)
print("Successfully indexed into Elasticsearch:", success, "| Failed:", failed)

Successfully indexed into Elasticsearch: 8 | Failed: []


In [6]:
# ---- Chroma (semantic side) ----
ALL_ACCESS_GROUPS = sorted({
    group
    for chunk in bytemage_documents
    for group in chunk["access_groups"]
})
print("Access groups seen in the corpus:", ALL_ACCESS_GROUPS)

try:
    client_chroma.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client_chroma.get_or_create_collection(name=COLLECTION_NAME)

ids, texts, embeddings, metadatas = [], [], [], []

for chunk in bytemage_documents:
    embedding_response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunk["text"])

    chroma_metadata = {
        "document_id": chunk["document_id"],
        "title": chunk["title"],
        "source": chunk["source"],
        "section": chunk["section"],
        "department": chunk["department"],
        "owner": chunk["owner"],
        "version": chunk["version"],
        "status": chunk["status"],
        "created_at": chunk["created_at"],
        "created_at_ts": chunk["created_at_ts"],
        "time_sensitive": chunk["time_sensitive"],
        "authoritative": chunk["authoritative"],
        "content_hash": chunk["content_hash"],
    }
    # Flatten the access_groups list into one boolean field per group.
    for group in ALL_ACCESS_GROUPS:
        chroma_metadata[f"access_{group}"] = group in chunk["access_groups"]

    ids.append(chunk["chunk_id"])
    texts.append(chunk["text"])
    embeddings.append(embedding_response.data[0].embedding)
    metadatas.append(chroma_metadata)

collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)
print(f"Indexed {collection.count()} documents into Chroma.")

Access groups seen in the corpus: ['employees', 'engineering', 'finance', 'leadership']
Indexed 8 documents into Chroma.


## 2. Add Metadata Filters to Retrieval

> Allow filters such as: source, document type, department, date range, version,
> owner, access group. Apply equivalent filters to both vector and lexical
> retrieval.

We use one generic `filters` dict as the shared vocabulary, then translate it
into each backend's native filter syntax — an Elasticsearch `bool` filter, and a
Chroma `where` clause. This mirrors Section 1's "consistent meaning,
store-appropriate encoding" idea: one filter dict in, equivalent results out of
either backend.

In [7]:
def build_es_filter(filters):
    """Translate the shared filter dict into Elasticsearch bool-query filter clauses."""
    clauses = []

    for field in ["department", "owner", "source", "version", "status", "document_id"]:
        if filters.get(field):
            clauses.append({"term": {field: filters[field]}})

    if filters.get("created_after_ts") is not None or filters.get("created_before_ts") is not None:
        range_clause = {}
        if filters.get("created_after_ts") is not None:
            range_clause["gte"] = filters["created_after_ts"]
        if filters.get("created_before_ts") is not None:
            range_clause["lte"] = filters["created_before_ts"]
        clauses.append({"range": {"created_at_ts": range_clause}})

    if filters.get("access_groups"):
        # "terms" matches if ANY element of the document's access_groups array
        # is present in the given list — exactly the "any overlap" semantics
        # access control needs.
        clauses.append({"terms": {"access_groups": filters["access_groups"]}})

    return clauses


def build_chroma_where(filters):
    """Translate the same shared filter dict into a Chroma `where` clause."""
    conditions = []

    for field in ["department", "owner", "source", "version", "status", "document_id"]:
        if filters.get(field):
            conditions.append({field: {"$eq": filters[field]}})

    if filters.get("created_after_ts") is not None:
        conditions.append({"created_at_ts": {"$gte": filters["created_after_ts"]}})

    if filters.get("created_before_ts") is not None:
        conditions.append({"created_at_ts": {"$lte": filters["created_before_ts"]}})

    if filters.get("access_groups"):
        access_conditions = [
            {f"access_{group}": {"$eq": True}}
            for group in filters["access_groups"]
            if group in ALL_ACCESS_GROUPS
        ]
        if len(access_conditions) == 1:
            conditions.append(access_conditions[0])
        elif len(access_conditions) > 1:
            conditions.append({"$or": access_conditions})

    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    return {"$and": conditions}

In [8]:
def make_result(chunk_id, text, metadata, rank, score, retriever):
    """Common retrieval result format — same shape as Step 7, plus the full
    metadata dict so downstream filtering/citation code has everything it needs."""
    return {
        "chunk_id": str(chunk_id),
        "text": text,
        "metadata": metadata,
        "rank": rank,
        "score": score,
        "retriever": retriever,
    }


def get_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def vector_search_filtered(query_text, filters=None, top_k=10):
    """Semantic retrieval via Chroma, with an optional metadata filter applied
    IN the query — unauthorized/filtered-out chunks never enter the result set."""
    t0 = time.perf_counter()

    query_embedding = get_embedding(query_text)
    where = build_chroma_where(filters or {})

    query_kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
        "include": ["documents", "distances", "metadatas"],
    }
    if where:
        query_kwargs["where"] = where

    raw = collection.query(**query_kwargs)

    ids = raw["ids"][0]
    texts = raw["documents"][0]
    distances = raw["distances"][0]
    metadatas = raw["metadatas"][0]

    results = [
        make_result(
            chunk_id=chunk_id,
            text=text,
            metadata=metadata,
            rank=i + 1,
            score=1 / (1 + distance),
            retriever="vector",
        )
        for i, (chunk_id, text, distance, metadata) in enumerate(
            zip(ids, texts, distances, metadatas)
        )
    ]

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms


def lexical_search_filtered(query_text, filters=None, top_k=10):
    """Lexical (BM25) retrieval via Elasticsearch, with the equivalent filter
    applied as native bool-query filter clauses."""
    t0 = time.perf_counter()

    query = {
        "size": top_k,
        "query": {
            "bool": {
                "must": {"match": {"text": query_text}},
                "filter": build_es_filter(filters or {}),
            }
        },
    }
    raw = es.search(index=INDEX_NAME, body=query)

    results = [
        make_result(
            chunk_id=hit["_source"]["chunk_id"],
            text=hit["_source"]["text"],
            metadata=hit["_source"],
            rank=i + 1,
            score=hit["_score"],
            retriever="bm25",
        )
        for i, hit in enumerate(raw["hits"]["hits"])
    ]

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms

In [9]:
# Demo: filter to Finance-owned documents only, regardless of query wording.
finance_only_results, _ = vector_search_filtered(
    "company policy",
    filters={"department": "Finance"},
    top_k=5,
)

print("Vector results filtered to department=Finance:")
for r in finance_only_results:
    print("-", r["chunk_id"], "|", r["metadata"]["title"], "-", r["metadata"]["section"])

# Same filter, lexical side — should match the same restriction.
finance_only_lexical, _ = lexical_search_filtered(
    "policy",
    filters={"department": "Finance"},
    top_k=5,
)

print("\nLexical results filtered to department=Finance:")
for r in finance_only_lexical:
    print("-", r["chunk_id"], "|", r["metadata"]["title"], "-", r["metadata"]["section"])

Vector results filtered to department=Finance:
- compensation-policy-001 | ByteMage Compensation Policy - Salary Bands

Lexical results filtered to department=Finance:


## 3. Enforce User Access Before Generation

> Associate the current user with access groups. Before retrieving context:
> (1) determine the user's allowed groups, (2) add access filters to the
> retrieval query, (3) exclude unauthorized chunks, (4) never rely on the model
> to ignore unauthorized content. Test with two users who have different
> permissions.

We reuse `merge_by_chunk_id` and `reciprocal_rank_fusion` exactly as built in
Step 7 — the only change is that `access_groups` is now **always** injected into
the filter dict before either retriever runs, so unauthorized chunks are
excluded at the retrieval layer and never reach fusion, citation assignment, or
the model. This is the "never rely on the model to ignore unauthorized content"
requirement: the model is never shown the content in the first place.

In [10]:
USERS = {
    "sarah_ahmed": {
        "full_name": "Sarah Ahmed",
        "role": "Engineering Manager",
        "access_groups": ["employees", "engineering", "leadership", "finance"],
    },
    "john_doe": {
        "full_name": "John Doe",
        "role": "Senior Software Engineer",
        "access_groups": ["employees", "engineering"],
    },
}


def get_user_allowed_groups(user_id):
    if user_id not in USERS:
        raise ValueError(f"Unknown user: {user_id}")
    return USERS[user_id]["access_groups"]


def merge_by_chunk_id(vector_results, lexical_results):
    """Merge vector + lexical result lists into one dict keyed by chunk_id.
    Identical logic to Step 7 Section 3, extended to carry the metadata dict."""
    merged = {}

    for rank, result in enumerate(vector_results, start=1):
        chunk_id = result["chunk_id"]
        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "metadata": result["metadata"],
                "vector_rank": None,
                "vector_score": None,
                "bm25_rank": None,
                "bm25_score": None,
            }
        merged[chunk_id]["vector_rank"] = rank
        merged[chunk_id]["vector_score"] = result["score"]

    for rank, result in enumerate(lexical_results, start=1):
        chunk_id = result["chunk_id"]
        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "metadata": result["metadata"],
                "vector_rank": None,
                "vector_score": None,
                "bm25_rank": None,
                "bm25_score": None,
            }
        merged[chunk_id]["bm25_rank"] = rank
        merged[chunk_id]["bm25_score"] = result["score"]

    return merged


def reciprocal_rank_fusion(merged, k=60):
    """RRF — identical to Step 7 Section 4."""
    results = []
    for entry in merged.values():
        vector_contribution = 1 / (k + entry["vector_rank"]) if entry["vector_rank"] is not None else 0
        bm25_contribution = 1 / (k + entry["bm25_rank"]) if entry["bm25_rank"] is not None else 0
        entry["rrf_score"] = vector_contribution + bm25_contribution
        results.append(entry)

    results.sort(key=lambda entry: entry["rrf_score"], reverse=True)
    return results


def hybrid_search_for_user(query_text, user_id, extra_filters=None, top_k=10):
    """Hybrid retrieval that ALWAYS injects the current user's access filter —
    there is no code path that skips it."""
    filters = dict(extra_filters or {})
    filters["access_groups"] = get_user_allowed_groups(user_id)

    vector_results, _ = vector_search_filtered(query_text, filters=filters, top_k=top_k)
    lexical_results, _ = lexical_search_filtered(query_text, filters=filters, top_k=top_k)

    merged = merge_by_chunk_id(vector_results, lexical_results)
    return reciprocal_rank_fusion(merged)

In [11]:
# Same query, two users with different access groups.
query = "What are the salary bands for senior engineers?"

sarah_results = hybrid_search_for_user(query, "sarah_ahmed", top_k=5)
john_results = hybrid_search_for_user(query, "john_doe", top_k=5)

print("Sarah Ahmed (has 'finance' group):")
for r in sarah_results:
    print("-", r["chunk_id"], "|", r["metadata"]["title"])

print("\nJohn Doe (no 'finance' group):")
for r in john_results:
    print("-", r["chunk_id"], "|", r["metadata"]["title"])

print("\nCompensation policy visible to Sarah:", any(r["chunk_id"] == "compensation-policy-001" for r in sarah_results))
print("Compensation policy visible to John: ", any(r["chunk_id"] == "compensation-policy-001" for r in john_results))

Sarah Ahmed (has 'finance' group):
- compensation-policy-001 | ByteMage Compensation Policy
- engineering-handbook-001 | ByteMage Engineering Handbook
- product-roadmap-2026-001 | ByteMage Product Roadmap
- leave-policy-v1-001 | ByteMage Leave Policy
- data-retention-policy-001 | ByteMage Data Retention Policy
- leave-policy-v2-001 | ByteMage Leave Policy
- product-roadmap-2025-001 | ByteMage Product Roadmap

John Doe (no 'finance' group):
- engineering-handbook-001 | ByteMage Engineering Handbook
- product-roadmap-2026-001 | ByteMage Product Roadmap
- onboarding-guide-001 | ByteMage Onboarding Guide
- leave-policy-v1-001 | ByteMage Leave Policy
- data-retention-policy-001 | ByteMage Data Retention Policy
- leave-policy-v2-001 | ByteMage Leave Policy
- product-roadmap-2025-001 | ByteMage Product Roadmap

Compensation policy visible to Sarah: True
Compensation policy visible to John:  False


## 4. Add Source Numbers

> Assign stable source numbers after reranking: `[1] Employee Handbook, page 12`.
> Include the same numbers inside the context sent to the model.

Numbers are assigned once, in final ranked order, and reused everywhere
downstream — in the prompt context, in the answer's citation markers, and in the
rendered citation cards (Section 6). "Stable" means the model and the UI must
agree on what `[1]` refers to.

In [12]:
def assign_source_numbers(chunks):
    """Assign 1-based source numbers in ranked order. Returns a new list of
    dicts shaped for both prompt-building and UI rendering."""
    sources = []
    for i, chunk in enumerate(chunks, start=1):
        metadata = chunk["metadata"]
        sources.append({
            "number": i,
            "chunk_id": chunk["chunk_id"],
            "title": metadata["title"],
            "section": metadata.get("section"),
            "source_file": metadata.get("source"),
            "document_id": metadata.get("document_id"),
            "version": metadata.get("version"),
            "text": chunk["text"],
        })
    return sources


def format_sources_for_prompt(sources):
    """Format sources with the SAME numbers used in the citation markers."""
    lines = []
    for source in sources:
        lines.append(f"[{source['number']}] {source['title']} — {source['section']}\n{source['text']}")
    return "\n\n".join(lines)

## 5. Require Citations in Generated Answers

> Prompt the model to cite factual claims using the provided source numbers, e.g.
> "Employees may take up to five sick days without additional approval [1]." Tell
> the model not to invent source numbers.

The prompt is explicit about three things: only use the numbered sources, cite
every factual claim, and never invent a number that isn't in the list. Section 7
then checks that the model actually followed the last rule — a system prompt is
an instruction, not a guarantee.

In [13]:
CITATION_SYSTEM_PROMPT = """You are a support assistant answering questions using ONLY the numbered sources below.

Rules:
- Cite the source number for every factual claim, like this: "...five sick days [1]."
- Only use source numbers that appear in the provided list below. NEVER invent a source number.
- If the sources do not contain enough information to answer, say so explicitly instead of guessing.
- If two sources give conflicting information, state both values and say they conflict, citing each one.

Sources:
{sources_block}
"""


def answer_with_citations(user_question, sources):
    """Generate an answer that must cite the given, pre-numbered sources."""
    system_prompt = CITATION_SYSTEM_PROMPT.format(sources_block=format_sources_for_prompt(sources))

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
    )

    return response.choices[0].message.content

In [14]:
query = "What is ByteMage's current top product priority?"

fused = hybrid_search_for_user(query, "john_doe", top_k=5)
sources = assign_source_numbers(fused)
answer = answer_with_citations(query, sources)

print("ANSWER:")
print(answer)

ANSWER:
ByteMage's current top product priority, as of Q3 2026, is launching the AI Search Platform's new billing dashboard for enterprise customers [2].


## 6. Render Citations in the Interface

> Below the answer, show: source number, document title, page or section,
> retrieved passage, file name or link. Make citation markers clickable where
> possible.

This notebook prints a plain-text citation card per source. In a real UI, the
`link` field below is what `[1]` would hyperlink to.

In [15]:
def render_citations(sources):
    """Print one citation card per source, in the format a UI would render
    below the generated answer."""
    for source in sources:
        link = f"internal-docs://{source['document_id']}/{source['chunk_id']}"
        print("=" * 70)
        print(f"[{source['number']}] {source['title']}")
        print(f"    Section:    {source['section']}")
        print(f"    File:       {source['source_file']}  (version {source['version']})")
        print(f"    Link:       {link}")
        print(f"    Passage:    {source['text'][:180]}{'...' if len(source['text']) > 180 else ''}")


print("ANSWER:")
print(answer)
print()
print("SOURCES:")
render_citations(sources)

ANSWER:
ByteMage's current top product priority, as of Q3 2026, is launching the AI Search Platform's new billing dashboard for enterprise customers [2].

SOURCES:
[1] ByteMage Product Roadmap
    Section:    Q1 Priorities
    File:       product-roadmap-2025.md  (version 2025-01)
    Link:       internal-docs://product-roadmap/product-roadmap-2025-001
    Passage:    As of Q1 2025, ByteMage's top product priority is migrating internal services to the new authentication system.
[2] ByteMage Product Roadmap
    Section:    Q3 Priorities
    File:       product-roadmap-2026.md  (version 2026-07)
    Link:       internal-docs://product-roadmap/product-roadmap-2026-001
    Passage:    As of Q3 2026, ByteMage's top product priority is launching the AI Search Platform's new billing dashboard for enterprise customers.
[3] ByteMage Onboarding Guide
    Section:    First Week
    File:       onboarding-guide.md  (version 2024-01)
    Link:       internal-docs://onboarding-guide/onboarding-guide

## 7. Validate Citation References

> After generation: extract citation markers from the answer, confirm each
> citation exists in the retrieved source list, flag unknown citations, flag
> answers with no citations when citations were required.

This is a pure text-processing check — no model call needed — so it runs on
every answer for free. It catches the failure mode the prompt in Section 5
merely *asked* the model to avoid: an invented `[7]` when only 5 sources exist.

In [16]:
def extract_citation_markers(answer_text):
    """Return the set of integers appearing as [n] markers in the answer."""
    return {int(match) for match in re.findall(r"\[(\d+)\]", answer_text)}


def validate_citations(answer_text, sources, require_citations=True):
    """Check that every citation marker in the answer refers to a real,
    provided source. Does NOT check whether the citation is factually
    accurate — that is Section 8's job."""
    cited_numbers = extract_citation_markers(answer_text)
    valid_numbers = {source["number"] for source in sources}

    unknown_citations = sorted(cited_numbers - valid_numbers)
    has_citations = len(cited_numbers) > 0

    return {
        "cited_numbers": sorted(cited_numbers),
        "unknown_citations": unknown_citations,
        "has_citations": has_citations,
        "missing_required_citations": require_citations and not has_citations,
        "is_valid": not unknown_citations and (has_citations or not require_citations),
    }

In [17]:
validation = validate_citations(answer, sources)
print("Citation validation on the real answer:")
print(validation)

# Demonstrate the failure case with a deliberately tampered answer that cites
# a source number that was never in the provided list.
tampered_answer = answer + " This is also confirmed in an internal memo [99]."
tampered_validation = validate_citations(tampered_answer, sources)
print("\nCitation validation on a tampered answer with an invented [99]:")
print(tampered_validation)

Citation validation on the real answer:
{'cited_numbers': [2], 'unknown_citations': [], 'has_citations': True, 'missing_required_citations': False, 'is_valid': True}

Citation validation on a tampered answer with an invented [99]:
{'cited_numbers': [2, 99], 'unknown_citations': [99], 'has_citations': True, 'missing_required_citations': False, 'is_valid': False}


## 8. Add Source-Support Checking

> For each sentence containing a citation: pair the sentence with the cited
> passage, ask a model or evaluator whether the passage supports the claim.
> Record: Supported, Partially supported, Unsupported. Do not treat citation
> presence as proof of citation correctness.

Section 7 proved the citation *number* is real. This section checks whether the
cited passage actually *says* what the sentence claims — a valid citation can
still misrepresent its source. We use structured output (`response_format`) so
the verdict is a constrained enum, not free text we'd have to re-parse.

In [18]:
from typing import Literal


class SupportCheck(BaseModel):
    verdict: Literal["supported", "partially_supported", "unsupported"]
    reason: str = Field(description="One sentence explaining the verdict.")


def split_into_sentences(text):
    """Very simple sentence splitter. Good enough for short generated answers;
    a real system would use a proper sentence tokenizer for edge cases like
    abbreviations and decimal numbers."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def check_sentence_support(sentence, cited_passages):
    """Ask the model whether the cited passage(s) actually support this one
    sentence's claim."""
    passages_block = "\n\n".join(cited_passages)

    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You check whether a source passage supports a claim. "
                    "Judge ONLY the given passage(s) — do not use outside knowledge."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Claim:\n{sentence}\n\n"
                    f"Cited passage(s):\n{passages_block}\n\n"
                    "Does the passage support the claim?"
                ),
            },
        ],
        response_format=SupportCheck,
    )

    return response.choices[0].message.parsed


def check_source_support(answer_text, sources):
    """Run support-checking on every sentence that carries a citation marker."""
    sources_by_number = {source["number"]: source for source in sources}
    results = []

    for sentence in split_into_sentences(answer_text):
        cited_numbers = extract_citation_markers(sentence)
        if not cited_numbers:
            continue

        cited_passages = [
            sources_by_number[n]["text"]
            for n in cited_numbers
            if n in sources_by_number
        ]
        if not cited_passages:
            # Already caught by validate_citations — nothing to support-check.
            continue

        check = check_sentence_support(sentence, cited_passages)
        results.append({
            "sentence": sentence,
            "cited_numbers": sorted(cited_numbers),
            "verdict": check.verdict,
            "reason": check.reason,
        })

    return results

In [19]:
support_results = check_source_support(answer, sources)

print("Source-support check (do not confuse this with citation validity from Section 7):")
for result in support_results:
    print("-" * 70)
    print("Sentence:", result["sentence"])
    print("Cited:   ", result["cited_numbers"])
    print("Verdict: ", result["verdict"], "-", result["reason"])

Source-support check (do not confuse this with citation validity from Section 7):
----------------------------------------------------------------------
Sentence: ByteMage's current top product priority, as of Q3 2026, is launching the AI Search Platform's new billing dashboard for enterprise customers [2].
Cited:    [2]
Verdict:  supported - The passage explicitly states that as of Q3 2026, ByteMage's top product priority is launching the AI Search Platform's new billing dashboard for enterprise customers, directly supporting the claim.


## 9. Handle Document Versions

> Store multiple versions of the same document. Add retrieval rules such as:
> prefer the newest active version, exclude archived versions by default, allow
> historical queries to retrieve older versions. Test contradictory policies
> across versions.

Our `leave-policy` document has two versions that genuinely disagree: v1
(archived, "three sick days") and v2 (active, "five sick days"). `status` is a
**hard filter** — archived versions are excluded unless the caller explicitly
asks for history — which is different from Section 10's freshness *ranking*,
where both versions of `product-roadmap` stay `"active"` and recency is only a
soft preference. Version status answers "is this allowed to be retrieved at
all?"; freshness answers "how should equally-allowed results be ordered?"

In [20]:
def hybrid_search_with_versioning(query_text, user_id, include_historical=False, top_k=10):
    """Same as hybrid_search_for_user, but defaults to active-only documents.
    Pass include_historical=True for the rare case an archived version is
    explicitly relevant (e.g. "what did the old policy say?")."""
    extra_filters = {} if include_historical else {"status": "active"}
    return hybrid_search_for_user(query_text, user_id, extra_filters=extra_filters, top_k=top_k)


query = "How many sick days are employees allowed?"

current_only = hybrid_search_with_versioning(query, "john_doe", include_historical=False)
print("Default (active only):")
for r in current_only:
    print("-", r["chunk_id"], "| version", r["metadata"]["version"], "| status", r["metadata"]["status"])

with_history = hybrid_search_with_versioning(query, "john_doe", include_historical=True)
print("\nWith include_historical=True:")
for r in with_history:
    print("-", r["chunk_id"], "| version", r["metadata"]["version"], "| status", r["metadata"]["status"])

Default (active only):
- leave-policy-v2-001 | version 2025-09 | status active
- onboarding-guide-001 | version 2024-01 | status active
- data-retention-policy-001 | version 2019-05 | status active
- engineering-handbook-001 | version 2025-07 | status active
- product-roadmap-2025-001 | version 2025-01 | status active
- product-roadmap-2026-001 | version 2026-07 | status active

With include_historical=True:
- leave-policy-v2-001 | version 2025-09 | status active
- leave-policy-v1-001 | version 2023-02 | status archived
- onboarding-guide-001 | version 2024-01 | status active
- data-retention-policy-001 | version 2019-05 | status active
- engineering-handbook-001 | version 2025-07 | status active
- product-roadmap-2025-001 | version 2025-01 | status active
- product-roadmap-2026-001 | version 2026-07 | status active


## 10. Add Freshness Rules

> Use metadata to prefer or require recent documents for time-sensitive topics.
> Do not apply freshness blindly — some authoritative documents may be old but
> still valid.

Freshness is scored 0-1 by exponential decay from `created_at`, **except**
chunks flagged `authoritative=True`, which always score `1.0` — old-but-still-
correct policies (our 2019 data-retention doc) are never penalized for age.

RRF scores and freshness scores live on different scales, exactly like BM25 and
cosine-similarity scores did in Step 7 Section 5 — so we normalize the RRF
score to `[0, 1]` before blending, instead of adding raw numbers together.

In [21]:
def freshness_score(metadata, now_ts, half_life_days=180):
    """1.0 for authoritative documents (freshness-exempt). Otherwise, exponential
    decay: the score halves every `half_life_days` days of age."""
    if metadata.get("authoritative"):
        return 1.0

    age_days = (now_ts - metadata["created_at_ts"]) / 86400
    return 0.5 ** (age_days / half_life_days)


def apply_freshness_boost(fused_results, freshness_weight=0.3, half_life_days=180):
    """Blend normalized RRF score with freshness score. freshness_weight=0
    recovers plain RRF ordering; freshness_weight=1 ignores relevance entirely."""
    if not fused_results:
        return fused_results

    now_ts = int(time.time())
    rrf_scores = [entry["rrf_score"] for entry in fused_results]
    min_rrf, max_rrf = min(rrf_scores), max(rrf_scores)
    rrf_range = max_rrf - min_rrf

    boosted = []
    for entry in fused_results:
        normalized_rrf = (entry["rrf_score"] - min_rrf) / rrf_range if rrf_range > 0 else 1.0
        freshness = freshness_score(entry["metadata"], now_ts, half_life_days)
        boosted_score = (1 - freshness_weight) * normalized_rrf + freshness_weight * freshness

        boosted.append({**entry, "freshness_score": freshness, "boosted_score": boosted_score})

    boosted.sort(key=lambda entry: entry["boosted_score"], reverse=True)
    return boosted

In [22]:
# Time-sensitive topic: the two product-roadmap versions are equally "about"
# the query, so freshness should decide the order.
query = "What is ByteMage working on right now?"
fused = hybrid_search_with_versioning(query, "john_doe", include_historical=False, top_k=10)
boosted = apply_freshness_boost(fused, freshness_weight=0.4)

print("Freshness-boosted ranking for a time-sensitive query:")
for r in boosted:
    if r["metadata"]["document_id"] == "product-roadmap":
        print(
            f"- {r['chunk_id']} | version {r['metadata']['version']} "
            f"| freshness={r['freshness_score']:.3f} | boosted_score={r['boosted_score']:.3f}"
        )

# Old-but-authoritative: the 2019 retention policy should NOT be penalized.
query = "How long does ByteMage retain customer data?"
fused = hybrid_search_with_versioning(query, "john_doe", include_historical=False, top_k=10)
boosted = apply_freshness_boost(fused, freshness_weight=0.4)

print("\nFreshness-boosted ranking for an authoritative-but-old document:")
for r in boosted:
    print(
        f"- {r['chunk_id']} | created {r['metadata']['created_at']} "
        f"| authoritative={r['metadata']['authoritative']} "
        f"| freshness={r['freshness_score']:.3f} | boosted_score={r['boosted_score']:.3f}"
    )

Freshness-boosted ranking for a time-sensitive query:
- product-roadmap-2026-001 | version 2026-07 | freshness=0.893 | boosted_score=0.800
- product-roadmap-2025-001 | version 2025-01 | freshness=0.103 | boosted_score=0.641

Freshness-boosted ranking for an authoritative-but-old document:
- data-retention-policy-001 | created 2019-05-01 | authoritative=True | freshness=1.000 | boosted_score=1.000
- leave-policy-v2-001 | created 2025-09-01 | authoritative=False | freshness=0.258 | boosted_score=0.658
- engineering-handbook-001 | created 2025-07-15 | authoritative=False | freshness=0.215 | boosted_score=0.623
- onboarding-guide-001 | created 2024-01-10 | authoritative=False | freshness=0.026 | boosted_score=0.583
- product-roadmap-2026-001 | created 2026-07-20 | authoritative=False | freshness=0.893 | boosted_score=0.357
- product-roadmap-2025-001 | created 2025-01-05 | authoritative=False | freshness=0.103 | boosted_score=0.060


## 11. Add Insufficient-Evidence Behavior

> Tell the model to distinguish between: directly supported answer, reasonable
> inference, information not found, conflicting sources. Provide a consistent
> response format for these cases.

We classify evidence from the **pipeline's own signals** — retrieval results,
version conflicts, support-check verdicts — rather than trusting the model to
self-report its own confidence. A model that hallucinates a citation is not a
reliable narrator of "how well-supported" its own answer is.

In [23]:
def detect_version_conflict(chunks):
    """Flag when top-ranked chunks include more than one version of the same
    document_id — a signal the retrieved context may contradict itself."""
    by_document = {}
    for chunk in chunks:
        document_id = chunk["metadata"]["document_id"]
        by_document.setdefault(document_id, []).append(chunk)

    return {
        document_id: entries
        for document_id, entries in by_document.items()
        if len({entry["metadata"]["version"] for entry in entries}) > 1
    }


def classify_evidence(retrieved_chunks, support_results, version_conflicts):
    """Map pipeline signals onto the roadmap's four evidence categories."""
    if not retrieved_chunks:
        return "information_not_found"

    if version_conflicts:
        return "conflicting_sources"

    if not support_results:
        # No cited sentences to check — the answer likely declined to answer,
        # or made claims without citing anything (caught separately by
        # validate_citations' missing_required_citations flag).
        return "reasonable_inference"

    verdicts = {result["verdict"] for result in support_results}
    if verdicts == {"supported"}:
        return "directly_supported"

    return "reasonable_inference"

## Putting It Together: A Governed Answer Pipeline

`answer_with_governance()` chains every section above into one call:

1. Resolve the user's access groups (Section 3) and inject them into retrieval.
2. Hybrid-retrieve with version rules applied (Section 9).
3. Freshness-boost the fused ranking (Section 10).
4. Detect version conflicts among the top results (Section 11).
5. Assign stable source numbers (Section 4) and generate a cited answer (Section 5).
6. Validate the citations (Section 7) and check source support (Section 8) —
   support-checking costs one model call per cited sentence, so it's a toggle,
   not something you'd necessarily run on every production request.
7. Classify the overall evidence category (Section 11) and render the result
   (Section 6).

If access filtering leaves zero authorized results, the function returns early
with a generic "not found in your sources" message — it never reveals that a
restricted document *exists* but was hidden, which would itself leak
information to an unauthorized user.

In [24]:
def answer_with_governance(
    user_id,
    query,
    include_historical=False,
    freshness_weight=0.3,
    top_k=5,
    run_support_check=True,
):
    fused = hybrid_search_with_versioning(query, user_id, include_historical=include_historical, top_k=10)
    boosted = apply_freshness_boost(fused, freshness_weight=freshness_weight)
    top_chunks = boosted[:top_k]

    if not top_chunks:
        return {
            "user_id": user_id,
            "query": query,
            "answer": "I couldn't find anything in the sources you have access to.",
            "sources": [],
            "citation_validation": None,
            "support_results": [],
            "version_conflicts": {},
            "evidence_class": "information_not_found",
        }

    version_conflicts = detect_version_conflict(top_chunks)
    sources = assign_source_numbers(top_chunks)
    answer = answer_with_citations(query, sources)

    citation_validation = validate_citations(answer, sources)
    support_results = check_source_support(answer, sources) if run_support_check else []

    evidence_class = classify_evidence(top_chunks, support_results, version_conflicts)

    return {
        "user_id": user_id,
        "query": query,
        "answer": answer,
        "sources": sources,
        "citation_validation": citation_validation,
        "support_results": support_results,
        "version_conflicts": version_conflicts,
        "evidence_class": evidence_class,
    }


def print_governed_result(result):
    print("=" * 78)
    print("USER: ", result["user_id"], "| QUERY:", result["query"])
    print("-" * 78)
    print("ANSWER:", result["answer"])
    print()
    print("Evidence class:      ", result["evidence_class"])
    if result["citation_validation"] is not None:
        print("Citation validation: ", result["citation_validation"])
    if result["version_conflicts"]:
        print("Version conflict across:", list(result["version_conflicts"].keys()))
    if result["support_results"]:
        unsupported = [r for r in result["support_results"] if r["verdict"] != "supported"]
        if unsupported:
            print(f"WARNING: {len(unsupported)} cited sentence(s) not fully supported by their source:")
            for r in unsupported:
                print("   -", r["sentence"], "->", r["verdict"])
    if result["sources"]:
        print()
        render_citations(result["sources"])

In [25]:
# Scenario 1: authorized user gets a direct, cited answer.
print_governed_result(answer_with_governance("sarah_ahmed", "What are the salary bands for senior engineers?"))

USER:  sarah_ahmed | QUERY: What are the salary bands for senior engineers?
------------------------------------------------------------------------------
ANSWER: ByteMage salary bands are reviewed every March. Specifically, Senior Software Engineers in the AI Platform department fall in Band E5 [3].

Evidence class:       conflicting_sources
Citation validation:  {'cited_numbers': [3], 'unknown_citations': [], 'has_citations': True, 'missing_required_citations': False, 'is_valid': True}
Version conflict across: ['product-roadmap']

[1] ByteMage Product Roadmap
    Section:    Q3 Priorities
    File:       product-roadmap-2026.md  (version 2026-07)
    Link:       internal-docs://product-roadmap/product-roadmap-2026-001
    Passage:    As of Q3 2026, ByteMage's top product priority is launching the AI Search Platform's new billing dashboard for enterprise customers.
[2] ByteMage Data Retention Policy
    Section:    Customer Data
    File:       data-retention-policy.md  (version 2019-

In [26]:
# Scenario 2: unauthorized user asking the exact same question — access-filtered
# to zero results, WITHOUT confirming that a restricted document exists.
print_governed_result(answer_with_governance("john_doe", "What are the salary bands for senior engineers?"))

USER:  john_doe | QUERY: What are the salary bands for senior engineers?
------------------------------------------------------------------------------
ANSWER: The provided sources do not contain any information regarding the salary bands for senior engineers.

Evidence class:       conflicting_sources
Citation validation:  {'cited_numbers': [], 'unknown_citations': [], 'has_citations': False, 'missing_required_citations': True, 'is_valid': False}
Version conflict across: ['product-roadmap']

[1] ByteMage Product Roadmap
    Section:    Q3 Priorities
    File:       product-roadmap-2026.md  (version 2026-07)
    Link:       internal-docs://product-roadmap/product-roadmap-2026-001
    Passage:    As of Q3 2026, ByteMage's top product priority is launching the AI Search Platform's new billing dashboard for enterprise customers.
[2] ByteMage Data Retention Policy
    Section:    Customer Data
    File:       data-retention-policy.md  (version 2019-05)
    Link:       internal-docs://data-

In [27]:
# Scenario 3: version handling — default excludes the archived policy, so the
# answer is unambiguous even though a contradictory old version exists.
print_governed_result(answer_with_governance("john_doe", "How many sick days are employees allowed?", include_historical=False))

USER:  john_doe | QUERY: How many sick days are employees allowed?
------------------------------------------------------------------------------
ANSWER: ByteMage employees may take up to five sick days per month without additional approval [1].

Evidence class:       directly_supported
Citation validation:  {'cited_numbers': [1], 'unknown_citations': [], 'has_citations': True, 'missing_required_citations': False, 'is_valid': True}

[1] ByteMage Leave Policy
    Section:    Sick Leave
    File:       leave-policy-v2.md  (version 2025-09)
    Link:       internal-docs://leave-policy/leave-policy-v2-001
    Passage:    ByteMage employees may take up to five sick days per month without additional approval. Extended sick leave requires notifying HR within 48 hours.
[2] ByteMage Onboarding Guide
    Section:    First Week
    File:       onboarding-guide.md  (version 2024-01)
    Link:       internal-docs://onboarding-guide/onboarding-guide-001
    Passage:    New ByteMage employees complet

In [28]:
# Scenario 4: same question, but explicitly including historical versions —
# now both the archived and active leave policy are retrieved, so the pipeline
# should flag conflicting_sources instead of picking one silently.
print_governed_result(answer_with_governance("john_doe", "How many sick days are employees allowed?", include_historical=True))

USER:  john_doe | QUERY: How many sick days are employees allowed?
------------------------------------------------------------------------------
ANSWER: There are two conflicting sources regarding the number of sick days ByteMage employees are allowed. One source states employees may take up to five sick days per month without additional approval [1], while another source states employees may take up to three sick days per month without additional approval [2].

Evidence class:       conflicting_sources
Citation validation:  {'cited_numbers': [1, 2], 'unknown_citations': [], 'has_citations': True, 'missing_required_citations': False, 'is_valid': True}
Version conflict across: ['leave-policy']

[1] ByteMage Leave Policy
    Section:    Sick Leave
    File:       leave-policy-v2.md  (version 2025-09)
    Link:       internal-docs://leave-policy/leave-policy-v2-001
    Passage:    ByteMage employees may take up to five sick days per month without additional approval. Extended sick leave 

In [29]:
# Scenario 5: freshness in action — the newer roadmap entry should win.
print_governed_result(answer_with_governance("john_doe", "What is ByteMage working on right now?"))

USER:  john_doe | QUERY: What is ByteMage working on right now?
------------------------------------------------------------------------------
ANSWER: As of Q3 2026, ByteMage's top product priority is launching the AI Search Platform's new billing dashboard for enterprise customers [1].

Evidence class:       conflicting_sources
Citation validation:  {'cited_numbers': [1], 'unknown_citations': [], 'has_citations': True, 'missing_required_citations': False, 'is_valid': True}
Version conflict across: ['product-roadmap']

[1] ByteMage Product Roadmap
    Section:    Q3 Priorities
    File:       product-roadmap-2026.md  (version 2026-07)
    Link:       internal-docs://product-roadmap/product-roadmap-2026-001
    Passage:    As of Q3 2026, ByteMage's top product priority is launching the AI Search Platform's new billing dashboard for enterprise customers.
[2] ByteMage Product Roadmap
    Section:    Q1 Priorities
    File:       product-roadmap-2025.md  (version 2025-01)
    Link:       i

In [30]:
# Scenario 6: old-but-authoritative document should still be retrieved and
# answered from directly, despite being from 2019.
print_governed_result(answer_with_governance("john_doe", "How long does ByteMage retain customer support data?"))

USER:  john_doe | QUERY: How long does ByteMage retain customer support data?
------------------------------------------------------------------------------
ANSWER: ByteMage retains customer support data for a minimum of seven years under regulatory requirement RX-118 [1].

Evidence class:       directly_supported
Citation validation:  {'cited_numbers': [1], 'unknown_citations': [], 'has_citations': True, 'missing_required_citations': False, 'is_valid': True}

[1] ByteMage Data Retention Policy
    Section:    Customer Data
    File:       data-retention-policy.md  (version 2019-05)
    Link:       internal-docs://data-retention-policy/data-retention-policy-001
    Passage:    ByteMage retains customer support data for a minimum of seven years under regulatory requirement RX-118. This requirement has not changed since 2019.
[2] ByteMage Leave Policy
    Section:    Sick Leave
    File:       leave-policy-v2.md  (version 2025-09)
    Link:       internal-docs://leave-policy/leave-policy

### Reflection

- **Are model-generated citations automatically reliable?** No — Section 7 shows
  a model can emit a citation number that was never in the source list, and
  Section 8 shows even a *valid* citation number can be paired with a claim the
  passage doesn't actually support.
- **How can the application verify a source supports a sentence?** Split the
  answer into sentences, pair each cited sentence with its cited passage(s), and
  ask a separate, narrowly-scoped model call (or a human) to judge support —
  never trust the generation call to grade its own work.
- **Should access restrictions be enforced in the prompt or retrieval layer?**
  Retrieval. A prompt instruction is advice the model can ignore or be
  jailbroken past; a retrieval filter means the unauthorized text is never in
  context at all. Section 3 enforces this by injecting `access_groups` into
  every retrieval call, with no code path that skips it.
- **What happens when two document versions contradict each other?** Detect the
  conflict from metadata (same `document_id`, different `version`, both
  retrieved) and surface it explicitly, rather than letting fusion/generation
  blend or silently pick one (Section 11, Scenario 4).
- **Should the newest document always outrank the most relevant one?** No —
  that's why freshness is a *weighted blend* with normalized relevance
  (Section 10), not a hard sort key, and why `authoritative` documents opt out
  of the freshness penalty entirely.
- **How should deleted documents be removed from all indexes?** They need a
  delete/soft-delete step against every store that indexed them (ES, Chroma,
  and any relational metadata store) — this notebook doesn't implement deletion,
  but the same `document_id` is the join key you'd delete by everywhere.
- **What is the difference between provenance and relevance?** Provenance is
  *where a chunk came from and whether it's trustworthy* (owner, version,
  status, authoritative) — Sections 1, 3, 9. Relevance is *how well it answers
  this query* (RRF score) — Step 7. A chunk can be highly relevant and low
  provenance (an archived, unauthorized draft) or low relevance and high
  provenance (an authoritative doc that's simply off-topic).
- **Should every sentence require a citation?** Only sentences making a factual
  claim from the sources — a sentence like "I don't have enough information for
  that" is correctly uncited. Section 8 only support-checks sentences that
  already carry a citation marker.
- **What makes a source authoritative?** Not age or length — it's an explicit
  editorial judgment captured as metadata (our `authoritative` flag), set by
  whoever owns the document, independent of how the freshness/relevance scoring
  happens to rank it.
- **How should the application present conflicting evidence?** Show both
  values, cite both sources, and say explicitly that they conflict — Section 5's
  prompt asks for exactly this, and Section 11's `conflicting_sources` class
  ensures the UI knows to render it as a conflict rather than a normal answer.